# Equal-States Ranking: QF vs LI

Compares quality_filtered (QF) vs logged_in_games (LI) when training on the top-K games
selected from a jittered ranking (pool_factor × K candidates, jittered with different seeds).
Each (dataset, K) pair trains on the same number of *states* (equal-states mode).

Parameters: pool_factor=1.5, K sweep=[1K, 2K, 5K, 10K, 20K, 30K], 5 random seeds per (dataset, K).
Capacity: 50L/50T for K≤5K, 100L/100T for K≥10K.

In [ ]:
import json
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

with open('ranking_li_vs_qf_no_drop.json') as f:
    data = json.load(f)

pool_factor = data['pool_factor']
k_sweep = data['k_sweep']
n_random = data['n_random']

# Group runs by (dataset, k) with seed extracted for pairing
groups = defaultdict(list)
for run in data['runs']:
    key = (run['dataset'], run['k'])
    seed = int(re.search(r'seed(\d+)', run['strategy']).group(1))
    groups[key].append({'seed': seed, 'pool_size': run['pool_size'], **run['metrics']})

# Sort each group by seed for consistent pairing
for key in groups:
    groups[key].sort(key=lambda r: r['seed'])

def extract_ranking_metric(groups, metric, ks=None):
    """Return K values, QF means/SEMs, LI means/SEMs for a metric."""
    if ks is None:
        ks = k_sweep
    out_ks, qf_m, qf_s, li_m, li_s = [], [], [], [], []
    for k in ks:
        qf_vals = [r[metric] for r in groups[('quality_filtered', k)]]
        li_vals = [r[metric] for r in groups[('logged_in_games', k)]]
        n = len(qf_vals)
        out_ks.append(k)
        qf_m.append(np.mean(qf_vals))
        qf_s.append(np.std(qf_vals) / np.sqrt(n))
        li_m.append(np.mean(li_vals))
        li_s.append(np.std(li_vals) / np.sqrt(n))
    return out_ks, np.array(qf_m), np.array(qf_s), np.array(li_m), np.array(li_s)

print(f'pool_factor={pool_factor}, k_sweep={k_sweep}, n_random={n_random}')
print(f'{len(groups)} groups, {len(data["runs"])} total runs')

## Summary Table

In [ ]:
def print_table(groups, k_sweep):
    print(f'{"Dataset":>18} {"K":>6} {"Pool":>6} {"States":>10}  '
          f'{"Loss":>18}  {"AUC":>18}  '
          f'{"Acc":>18}  {"Egg":>18}  {"SymDev":>18}')
    print('-' * 130)
    for ds in ['quality_filtered', 'logged_in_games']:
        for k in k_sweep:
            runs = groups[(ds, k)]
            pool = runs[0]['pool_size']
            loss = [r['log_loss'] for r in runs]
            auc = [r['auc_roc'] for r in runs]
            acc = [r['accuracy'] for r in runs]
            egg = [r['egg_inversion_rate'] for r in runs]
            sym = [r['symmetry_deviation'] for r in runs]
            n_states = int(np.mean([r['n_states'] for r in runs]))
            print(f'{ds:>18} {k:>6} {pool:>6} {n_states:>10,}  '
                  f'{np.mean(loss):.4f} +/- {np.std(loss):.4f}  '
                  f'{np.mean(auc):.4f} +/- {np.std(auc):.4f}  '
                  f'{np.mean(acc):.4f} +/- {np.std(acc):.4f}  '
                  f'{np.mean(egg):.4f} +/- {np.std(egg):.4f}  '
                  f'{np.mean(sym):.4f} +/- {np.std(sym):.4f}')
        print()

print_table(groups, k_sweep)

## Scaling Plots

QF vs LI across K values, error bars = ±1 SEM over 5 seeds.

In [ ]:
metrics = [
    ('log_loss', 'Log Loss'),
    ('auc_roc', 'AUC-ROC'),
    ('accuracy', 'Accuracy'),
    ('egg_inversion_rate', 'Egg Inversion Rate'),
    ('symmetry_deviation', 'Symmetry Deviation'),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for ax, (metric_key, metric_label) in zip(axes, metrics):
    ks, qf_m, qf_s, li_m, li_s = extract_ranking_metric(groups, metric_key)

    ax.errorbar(ks, qf_m, yerr=qf_s,
                marker='o', capsize=4, label='QF', color='#2196F3')
    ax.errorbar(ks, li_m, yerr=li_s,
                marker='s', capsize=4, label='LI', color='#FF9800')

    ax.set_xlabel('K (games selected)')
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label)
    ax.set_xscale('log', base=2)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_minor_formatter(plt.NullFormatter())

    ax.set_xticks(ks)
    labels = [f'{k//1000}K' for k in ks]
    ax.set_xticklabels(labels, fontsize=8)

for ax in axes[len(metrics):]:
    ax.set_visible(False)

fig.suptitle(f'Equal-States Ranking: QF vs LI (pool_factor={pool_factor})\n'
             f'({n_random} seeds, error bars = ±1 SEM)',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('ranking_scaling_plot.png', dpi=200, bbox_inches='tight')
print('Saved ranking_scaling_plot.png')
plt.show()

## QF Advantage (Paired Delta Plots)

Per-seed QF-LI differences for tighter error bars (paired comparison).
Blue = QF better, Red = LI better.

In [ ]:
metrics_delta = [
    ('log_loss', 'Log Loss (QF - LI)', -1),
    ('auc_roc', 'AUC-ROC (QF - LI)', 1),
    ('accuracy', 'Accuracy (QF - LI)', 1),
    ('egg_inversion_rate', 'Egg Inversion (QF - LI)', -1),
    ('symmetry_deviation', 'Symmetry Dev (QF - LI)', -1),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for ax, (metric_key, metric_label, sign) in zip(axes, metrics_delta):
    delta_means = []
    delta_sems = []

    for k in k_sweep:
        qf_runs = groups[('quality_filtered', k)]
        li_runs = groups[('logged_in_games', k)]
        # Paired by seed
        deltas = [qf[metric_key] - li[metric_key]
                  for qf, li in zip(qf_runs, li_runs)]
        delta_means.append(np.mean(deltas))
        delta_sems.append(np.std(deltas) / np.sqrt(len(deltas)))

    delta_means = np.array(delta_means)
    delta_sems = np.array(delta_sems)

    colors = ['#2196F3' if (d * sign > 0) else '#FF5722' for d in delta_means]
    ax.bar(range(len(k_sweep)), delta_means, yerr=delta_sems,
           capsize=5, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(k_sweep)))
    labels = [f'{k//1000}K' for k in k_sweep]
    ax.set_xticklabels(labels)
    ax.set_xlabel('K (games selected)')
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label)
    ax.grid(True, alpha=0.3, axis='y')

    if sign == -1:
        ax.text(0.98, 0.02, '\u2193 QF better', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=8, color='#2196F3')
    else:
        ax.text(0.98, 0.98, '\u2191 QF better', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, color='#2196F3')

for ax in axes[len(metrics_delta):]:
    ax.set_visible(False)

fig.suptitle(f'QF Advantage — Equal-States (Paired by Seed, pool_factor={pool_factor})\n'
             f'Blue = QF better, Red = LI better (error bars = ±1 SEM)',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('ranking_delta_plot.png', dpi=200, bbox_inches='tight')
print('Saved ranking_delta_plot.png')
plt.show()

# Drop 90% States Experiment

Same equal-states ranking setup but with `drop_prob=0.9` — each state has 90% chance of being
dropped during materialization. This tests whether QF vs LI differences hold when training on
far fewer states per game. K=[5K, 10K, 20K, 40K, 80K], 100L/100T, 5 seeds.

In [ ]:
with open('ranking_li_vs_qf_drop90.json') as f:
    drop_data = json.load(f)

drop_k_sweep = drop_data['k_sweep']
drop_capacity = f"{drop_data['num_leaves']}L/{drop_data['num_trees']}T"

drop_groups = defaultdict(list)
for run in drop_data['runs']:
    key = (run['dataset'], run['k'])
    seed = int(re.search(r'seed(\d+)', run['strategy']).group(1))
    drop_groups[key].append({'seed': seed, 'pool_size': run['pool_size'], **run['metrics']})

for key in drop_groups:
    drop_groups[key].sort(key=lambda r: r['seed'])

def extract_drop_metric(groups, k_sweep, metric):
    ks, qf_m, qf_s, li_m, li_s = [], [], [], [], []
    for k in k_sweep:
        qf_vals = [r[metric] for r in groups[('quality_filtered', k)]]
        li_vals = [r[metric] for r in groups[('logged_in_games', k)]]
        n = len(qf_vals)
        ks.append(k)
        qf_m.append(np.mean(qf_vals))
        qf_s.append(np.std(qf_vals) / np.sqrt(n))
        li_m.append(np.mean(li_vals))
        li_s.append(np.std(li_vals) / np.sqrt(n))
    return ks, np.array(qf_m), np.array(qf_s), np.array(li_m), np.array(li_s)

print(f'drop_prob={drop_data["drop_prob"]}, capacity={drop_capacity}')
print(f'k_sweep={drop_k_sweep}, n_random={drop_data["n_random"]}')
print(f'{len(drop_groups)} groups, {len(drop_data["runs"])} total runs')

## Drop90 Summary Table

In [ ]:
print_table(drop_groups, drop_k_sweep)

## No-Drop vs Drop90 Comparison

Overlay both experiments on shared K values (5K, 10K, 20K).
Solid = no drop, dashed = drop90. Blue = QF, orange = LI.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for ax, (metric_key, metric_label) in zip(axes, metrics):
    # No-drop (solid) — full K range
    nd_ks, nd_qf_m, nd_qf_s, nd_li_m, nd_li_s = extract_ranking_metric(groups, metric_key)
    ax.errorbar(nd_ks, nd_qf_m, yerr=nd_qf_s,
                marker='o', capsize=4, label='QF (no drop)', color='#2196F3')
    ax.errorbar(nd_ks, nd_li_m, yerr=nd_li_s,
                marker='s', capsize=4, label='LI (no drop)', color='#FF9800')

    # Drop90 (dashed) — full K range
    d_ks, d_qf_m, d_qf_s, d_li_m, d_li_s = extract_ranking_metric(
        drop_groups, metric_key, ks=drop_k_sweep)
    ax.errorbar(d_ks, d_qf_m, yerr=d_qf_s,
                marker='o', capsize=4, label='QF (drop90)', color='#2196F3',
                linestyle='--', alpha=0.6)
    ax.errorbar(d_ks, d_li_m, yerr=d_li_s,
                marker='s', capsize=4, label='LI (drop90)', color='#FF9800',
                linestyle='--', alpha=0.6)

    ax.set_xlabel('K (games selected)')
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label)
    ax.set_xscale('log', base=2)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_minor_formatter(plt.NullFormatter())

    all_ks = sorted(set(k_sweep) | set(drop_k_sweep))
    ax.set_xticks(all_ks)
    ax.set_xticklabels([f'{k//1000}K' for k in all_ks], fontsize=7)

for ax in axes[len(metrics):]:
    ax.set_visible(False)

fig.suptitle('No-Drop vs Drop90: QF vs LI (equal-states, 100L/100T)\n'
             'Solid = no drop, dashed = drop 90%',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('ranking_drop_comparison.png', dpi=200, bbox_inches='tight')
print('Saved ranking_drop_comparison.png')
plt.show()

## Drop90 QF Advantage (Paired Delta Plots)

Per-seed QF-LI differences for the drop90 experiment.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for ax, (metric_key, metric_label, sign) in zip(axes, metrics_delta):
    delta_means = []
    delta_sems = []

    for k in drop_k_sweep:
        qf_runs = drop_groups[('quality_filtered', k)]
        li_runs = drop_groups[('logged_in_games', k)]
        deltas = [qf[metric_key] - li[metric_key]
                  for qf, li in zip(qf_runs, li_runs)]
        delta_means.append(np.mean(deltas))
        delta_sems.append(np.std(deltas) / np.sqrt(len(deltas)))

    delta_means = np.array(delta_means)
    delta_sems = np.array(delta_sems)

    colors = ['#2196F3' if (d * sign > 0) else '#FF5722' for d in delta_means]
    ax.bar(range(len(drop_k_sweep)), delta_means, yerr=delta_sems,
           capsize=5, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(drop_k_sweep)))
    ax.set_xticklabels([f'{k//1000}K' for k in drop_k_sweep])
    ax.set_xlabel('K (games selected)')
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label)
    ax.grid(True, alpha=0.3, axis='y')

    if sign == -1:
        ax.text(0.98, 0.02, '\u2193 QF better', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=8, color='#2196F3')
    else:
        ax.text(0.98, 0.98, '\u2191 QF better', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, color='#2196F3')

for ax in axes[len(metrics_delta):]:
    ax.set_visible(False)

fig.suptitle(f'QF Advantage — Drop90 Equal-States ({drop_capacity}, pool_factor={pool_factor})\n'
             f'Blue = QF better, Red = LI better (error bars = \u00b11 SEM)',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('ranking_drop90_delta_plot.png', dpi=200, bbox_inches='tight')
print('Saved ranking_drop90_delta_plot.png')
plt.show()